In [ ]:
#importing libraries 

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#we will use flan-t5 as it is a small model that can run locally and its fast

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    trust_remote_code=False,
).to(device)

c:\Users\miqi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1498.22it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
#reading cleaned data

df = pd.read_csv("https://raw.githubusercontent.com/smith8jas/DeepLearning_BookRecommender/main/data/processed/cleaned_books.csv")
print(df.columns.tolist())

['title', 'authors', 'categories', 'description', 'average_rating', 'ratings_count', 'published_year', 'clean_title', 'clean_authors', 'clean_categories', 'clean_description', 'popularity_score', 'combined_text']


In [ ]:
def recommend(user_query, top_k=5): #top K limits how many books you pass to the model
    
    #mask filters the dataset to only books that match the user query
    
    mask = df["clean_categories"].str.contains(user_query, case=False, na=False)
    candidates = df[mask].head(top_k)

    if candidates.empty: #fallback if no books match and pick a random sample
        print("No books found for that query, showing random recommendations.")
        candidates = df.sample(top_k)

    books_text = "" #readibility purpose
    for _, row in candidates.iterrows():
        books_text += f"- {row['title']} by {row['authors']} ({row['categories']})\n"

    prompt = f"""You are a book expert. A user wants books about: {user_query}
    Here are some candidates:
    {books_text}
    Recommend the best ones and explain why.""" #model prompts

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device) #tokenitzation of the prompt in order to feed the model
    outputs = model.generate(**inputs, max_new_tokens=200) #max_new_tokens in order to not have long outputs
    return tokenizer.decode(outputs[0], skip_special_tokens=True) #back to letters

user_query = input("What kind of book are you looking for? ").strip() #user interaction
if len(user_query) > 100:
    print("Query too long, please be more specific.")
else:
    print(recommend(user_query))

No books found for that query, showing random recommendations.
A Time to Kill by John Grisham (Fiction)
